# 05 — Test market-data (résilience au restart container)

Smoke test du container `fxvol-market-data` — étape 5/5 (dernier de la série). Valide que **l'engine récupère proprement après un `docker restart`** : heartbeat repart en < 30s, ticks reprennent sur le pub/sub.

**Pourquoi c'est critique** : en prod, le container peut être restart pour plein de raisons (deploy d'une nouvelle image, OOM kill, restart Docker daemon, etc.). Le pipeline doit reprendre sans intervention humaine. Si la récupération est cassée, on perd des minutes de ticks à chaque incident → frontend figé pendant ce temps.

**Couvre** :
1. Baseline — heartbeat actuel existe (point de départ avant restart)
2. Restart container `fxvol-market-data` (subprocess `docker restart` — pas `docker compose restart` qui re-évaluerait le compose entier)
3. Attendre que **un nouveau** heartbeat apparaisse (timestamp ≠ baseline) en < 30s — preuve que l'engine s'est reconnecté à IB et a repris la boucle
4. Confirmer que les ticks pub/sub reprennent : SUBSCRIBE channel `ticks` après le restart, recevoir ≥ 3 messages en 5s
5. État final : heartbeat frais (< 5s), `latest_spot:EURUSD` non-stale

**Préreq** :
- Notebooks 01-04 verts
- ib-gateway healthy (sinon l'engine ne pourra pas reprendre la connexion IB après restart)
- Le restart prend ~10-30s, prévoir un peu de patience

**Note sur le restart d'ib-gateway** : tester `docker restart fxvol-ib-gateway` et vérifier que market-data backoff puis reprend, ce serait un test plus complet. **On le skip délibérément** : (a) le restart d'ib-gateway prend 60-90s à cause du login IBC, (b) ça monopolise la session IB pendant ce temps, (c) le scénario est déjà couvert par le backoff exponentiel de `src/shared/ib_connection.py` qui retry forever en prod. À tester manuellement si tu veux le valider une fois.

**Référence** : `src/shared/ib_connection.py` (backoff retries), `src/services/market_data/main.py` (signal handler SIGTERM), `src/services/market_data/engine.py` (graceful shutdown sur stop_event).

## Setup

In [8]:
import json
import subprocess
import time
from datetime import datetime, timezone

import redis

REDIS_URL = "redis://localhost:6380/0"
CONTAINER = "fxvol-market-data"
SYMBOL = "EURUSD"

# Délai max accepté entre le restart et la reprise du heartbeat.
# 30s = même fenêtre que le TTL Redis du heartbeat ; au-delà la clé
# expire et on a un trou côté frontend.
RECOVERY_DEADLINE_S = 30.0

# Pour confirmer que les ticks reprennent post-restart.
POST_RESTART_LISTEN_S = 5.0
MIN_POST_RESTART_MESSAGES = 3

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

r = redis.from_url(REDIS_URL, decode_responses=True)
if not r.ping():
    raise RuntimeError("Redis ping FAIL — check 'docker compose ps'")
print(f"Connected -> {REDIS_URL}")

Connected -> redis://localhost:6380/0


## 1. Baseline — heartbeat avant restart

**Ce que tu dois voir** : un heartbeat ISO-8601 frais (post notebooks 01-04 qui ont validé que market-data tourne). Si baseline absent → le container est déjà cassé, restart ne va pas aider — re-vérifie les notebooks 01/02 d'abord.

In [9]:
print("== 1. baseline heartbeat ==")

baseline_hb = r.get("heartbeat:market_data")
record("heartbeat baseline présent",
       baseline_hb is not None,
       baseline_hb if baseline_hb else "<missing>")

if baseline_hb is None:
    raise RuntimeError(
        "Pas de heartbeat baseline — container market-data non opérationnel.\n"
        "Vérifier `docker logs fxvol-market-data` et `notebook 01_test_container.ipynb`."
    )

== 1. baseline heartbeat ==
  [OK  ] heartbeat baseline présent  | 2026-04-28T14:39:13.391137Z


## 2. Restart container

On utilise `docker restart fxvol-market-data` (commande runtime Docker) plutôt que `docker compose restart market-data`. Pourquoi : `docker compose restart` re-évalue **tout le fichier compose** avant d'exécuter, y compris les `${VAR:?...}` du service ib-gateway. Si la PowerShell qui lance le notebook n'a pas chargé les secrets via `load_secrets.ps1` (cas typique : on a lancé Jupyter depuis PyCharm), compose fail avec `IB_PASSWORD is missing a value`. `docker restart` agit directement sur le container existant, sans re-parser le compose.

L'effet est strictement identique pour le test : SIGTERM → grace 10s → SIGKILL si pas mort → start. `src/services/market_data/main.py` a un signal handler qui set `stop_event` et fait un `ib.disconnect()` propre dans le finally. Si la signal handler est cassée, le container met 10s pour mourir → tu verras un délai un peu plus long ici.

In [10]:
print("== 2. docker restart fxvol-market-data ==")

t0 = time.perf_counter()
out = subprocess.run(
    ["docker", "restart", CONTAINER],
    capture_output=True, text=True,
)
elapsed = time.perf_counter() - t0

record("docker restart",
       out.returncode == 0,
       f"exit={out.returncode}, took {elapsed:.1f}s")

if out.stderr.strip():
    print(f"  [INFO] stderr: {out.stderr.strip()[:200]}")

== 2. docker restart fxvol-market-data ==
  [OK  ] docker restart  | exit=0, took 0.6s


## 3. Attendre un nouveau heartbeat (≠ baseline) en < 30s

**Ce que tu dois voir** : poll de `heartbeat:market_data` toutes les 0.5s. Quand on observe un timestamp différent de la baseline ET parsable ISO, on sait que l'engine a repris.

**Si timeout 30s** :
- engine ne reprend pas la connexion IB (ib-gateway down ou TrustedIPs ré-écrasés ?)
- engine crash au boot → `docker logs fxvol-market-data --tail 50`
- bug signal handler → engine reste en stop_event=True après le restart

In [11]:
print(f"== 3. attendre nouveau heartbeat ≤ {RECOVERY_DEADLINE_S}s ==")

t0 = time.perf_counter()
deadline = t0 + RECOVERY_DEADLINE_S
new_hb = None
while time.perf_counter() < deadline:
    current = r.get("heartbeat:market_data")
    if current and current != baseline_hb:
        try:
            datetime.fromisoformat(current.replace("Z", "+00:00"))
            new_hb = current
            break
        except ValueError:
            pass
    time.sleep(0.5)

recovery_s = time.perf_counter() - t0
record(f"nouveau heartbeat ≠ baseline en ≤ {RECOVERY_DEADLINE_S}s",
       new_hb is not None,
       f"recovered in {recovery_s:.1f}s — new = {new_hb!r}" if new_hb
       else f"timeout après {recovery_s:.1f}s — heartbeat toujours = {baseline_hb!r}")

== 3. attendre nouveau heartbeat ≤ 30.0s ==
  [OK  ] nouveau heartbeat ≠ baseline en ≤ 30.0s  | recovered in 2.5s — new = '2026-04-28T14:39:17.010504Z'


## 4. Les ticks reprennent sur le pub/sub

**Ce que tu dois voir** : SUBSCRIBE au channel `ticks` après le restart, ≥ 3 messages en 5s. Plancher plus bas que les notebooks 03/04 (5 messages) parce qu'on est juste après le restart, le ticker IB peut prendre 1-2s à se ré-amorcer.

**Si FAIL** :
- engine connecté mais ne publie pas → revoir `_publish_ticks_to_redis` dans `engine.py`
- IB pas reconnecté → cf. logs

In [12]:
print(f"== 4. ticks reprennent — listen {POST_RESTART_LISTEN_S}s ==")

sub = redis.from_url(REDIS_URL, decode_responses=True).pubsub(ignore_subscribe_messages=True)
sub.subscribe("ticks")

messages = []
deadline = time.perf_counter() + POST_RESTART_LISTEN_S
while time.perf_counter() < deadline:
    msg = sub.get_message(timeout=0.1)
    if msg and msg.get("type") == "message":
        try:
            json.loads(msg["data"])
            messages.append(msg["data"])
        except json.JSONDecodeError:
            pass

sub.unsubscribe("ticks")
sub.close()

record(f"≥ {MIN_POST_RESTART_MESSAGES} messages JSON post-restart",
       len(messages) >= MIN_POST_RESTART_MESSAGES,
       f"{len(messages)} messages")

== 4. ticks reprennent — listen 5.0s ==
  [OK  ] ≥ 3 messages JSON post-restart  | 25 messages


## 5. État final — heartbeat frais + `latest_spot` non-stale

**Ce que tu dois voir** : juste après les §3-§4, le state Redis est complètement frais. Si le heartbeat est déjà vieux de 5s à ce point, c'est qu'on est dans un crashloop intermittent (engine reboot, push 1 heartbeat, re-crash...). Tester aussi le `latest_spot` qui doit être présent — preuve que la boucle 100ms + push cache reste active.

In [13]:
print("== 5. état final ==")

current_hb = r.get("heartbeat:market_data")
if current_hb:
    try:
        hb_dt = datetime.fromisoformat(current_hb.replace("Z", "+00:00"))
        age_s = (datetime.now(timezone.utc) - hb_dt).total_seconds()
        record("heartbeat frais (age < 5s)",
               age_s < 5.0,
               f"age = {age_s:.2f}s")
    except ValueError as e:
        record("heartbeat parsable ISO", False, str(e))
else:
    record("heartbeat présent", False, "<missing>")

spot_raw = r.get(f"latest_spot:{SYMBOL}")
spot_ttl = r.ttl(f"latest_spot:{SYMBOL}")
record("latest_spot:EURUSD présent + TTL > 0",
       spot_raw is not None and spot_ttl > 0,
       f"value={spot_raw}, TTL={spot_ttl}s")

== 5. état final ==
  [OK  ] heartbeat frais (age < 5s)  | age = 0.28s
  [OK  ] latest_spot:EURUSD présent + TTL > 0  | value=1.1696300000000002, TTL=30s


## Récap final

In [14]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — market-data récupère proprement d'un restart.")
    print("Surface market-data complètement validée (notebooks 01-05).")
    print("Prochain container : frontend (le dernier de la migration).")


LABEL                                                   STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
heartbeat baseline présent                              OK      2026-04-28T14:39:13.391137Z
docker restart                                          OK      exit=0, took 0.6s
nouveau heartbeat ≠ baseline en ≤ 30.0s                 OK      recovered in 2.5s — new = '2026-04-28T14:39:17.010504Z'
≥ 3 messages JSON post-restart                          OK      25 messages
heartbeat frais (age < 5s)                              OK      age = 0.28s
latest_spot:EURUSD présent + TTL > 0                    OK      value=1.1696300000000002, TTL=30s
--------------------------------------------------------------------------------------------------------------

6 OK / 0 FAIL  (6 total)

OK — market-data récupère proprement d'un restart.
Surface market-data complètement validée (notebooks 01-05).
Prochain container : fro

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| baseline absent au §1 | container market-data déjà cassé avant le test | revoir notebooks 01/02 d'abord |
| restart prend > 15s au §2 | signal handler SIGTERM cassé, Docker doit SIGKILL | investiguer `src/services/market_data/main.py` |
| §3 timeout 30s | engine ne reprend pas la connexion IB | vérifier ib-gateway healthy, `docker logs fxvol-market-data` |
| §3 OK mais §4 0 messages | bridge api restart pas synchrone, ou throttle réinitialisé | re-lancer le notebook ; si persiste, investiguer pubsub |
| §3 OK + §4 OK mais §5 heartbeat age > 5s | crashloop : engine reboot, 1 push, re-crash | `docker logs fxvol-market-data --tail 100` ; chercher exceptions |
| `latest_spot` absent au §5 | engine connecté à IB mais ticks pas pushed | revoir engine.py, `_publish_ticks_to_redis` |